# Konvertering av Sampers OD-matriser (2040) till Dynameq-format

Denna notebook konverterar OD-matriser fran Sampers-format till Dynameq-format for prognosar 2040.

**Steg:**
1. Las zonkonverteringsnyckel (nycklar.xlsx)
2. Las befintliga Dynameq-matriser (DQT-format) for tidsprofil
3. Las och summera Sampers OD-matriser (CSV)
4. Konvertera zoner: Sampers -> Dynameq (med dag/nattbefolkningsfordelning)
5. Konvertera tid: Sampers 3h-total -> Dynameq kvartstimmatriser (14:00-18:00)
6. Skriv ut nya DQT-filer
7. Visualisering och rimlighetsanalys

In [ ]:
import pandas as pd
import numpy as np
import openpyxl
import os
import copy
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Bas-sokvag
BASE_DIR = Path(".")
DYNAMEQ_DIR = BASE_DIR / "Dynameq" / "matriser"
SAMPERS_LB_DIR = BASE_DIR / "Sampers" / "LB"
SAMPERS_PB_DIR = BASE_DIR / "Sampers" / "PB"
OUTPUT_DIR = BASE_DIR / "output_2040"
OUTPUT_DIR.mkdir(exist_ok=True)

print("Sokvagar konfigurerade.")
print(f"  Dynameq-matriser: {DYNAMEQ_DIR}")
print(f"  Sampers LB: {SAMPERS_LB_DIR}")
print(f"  Sampers PB: {SAMPERS_PB_DIR}")
print(f"  Output: {OUTPUT_DIR}")

## 1. Las zonkonverteringsnyckel

In [ ]:
# Las zonnyckeln fran Excel
key_df = pd.read_excel(BASE_DIR / "nycklar.xlsx")
key_df.columns = ["dynameq_zon", "sampers_zon", "kommentar", "nattbefolkning", "dagbefolkning"]

# Konvertera zoner till strangformat for konsekvent hantering
key_df["dynameq_zon"] = key_df["dynameq_zon"].astype(str)
key_df["sampers_zon"] = key_df["sampers_zon"].astype(str)

# Fixa kand bugg: 7381092 forekommer tva ganger, andra ska vara 7381093
duplicates = key_df[key_df.duplicated(subset=["dynameq_zon"], keep=False)]
if len(duplicates) > 0:
    print(f"Duplicerade Dynameq-zoner hittade:")
    print(duplicates)
    # Fixa: andra raden med 7381092 ska vara 7381093
    dup_idx = key_df[key_df["dynameq_zon"] == "7381092"].index
    if len(dup_idx) == 2:
        key_df.loc[dup_idx[1], "dynameq_zon"] = "7381093"
        print("-> Fixat: Andra '7381092' andrad till '7381093'")

# Filtrera bort zoner utan befolkningsdata (13, 14, 15 - specialzoner utan Sampers-data)
special_zones = key_df[key_df["nattbefolkning"].isna()]
print(f"\nSpecialzoner (utan befolkningsdata, hanteras separat):")
print(special_zones[["dynameq_zon", "sampers_zon", "kommentar"]])

# Arbetsnyckel: bara zoner med befolkningsdata
zone_key = key_df[key_df["nattbefolkning"].notna()].copy()
zone_key["nattbefolkning"] = zone_key["nattbefolkning"].astype(float)
zone_key["dagbefolkning"] = zone_key["dagbefolkning"].astype(float)

# Lagg till 738130 som saknas i nyckeln men finns i bade Dynameq och Sampers (1:1 mappning)
if "738130" not in zone_key["dynameq_zon"].values:
    new_row = pd.DataFrame({
        "dynameq_zon": ["738130"],
        "sampers_zon": ["738130"],
        "kommentar": ["Tillagd automatiskt - 1:1 mappning"],
        "nattbefolkning": [1.0],
        "dagbefolkning": [1.0]
    })
    zone_key = pd.concat([zone_key, new_row], ignore_index=True)
    print("\n-> Lagt till zon 738130 (1:1 mappning, saknas i nyckeln)")

# Visa sammanfattning
print(f"\nAntal zonmappningar: {len(zone_key)}")
print(f"Antal unika Sampers-zoner: {zone_key['sampers_zon'].nunique()}")
print(f"Antal unika Dynameq-zoner: {zone_key['dynameq_zon'].nunique()}")

# Kontrollera att befolkningsandelar summerar till ~1.0 per Sampers-zon
for col in ["nattbefolkning", "dagbefolkning"]:
    sums = zone_key.groupby("sampers_zon")[col].sum()
    bad = sums[~np.isclose(sums, 1.0, atol=0.05)]
    if len(bad) > 0:
        print(f"\nVARNING: {col} summerar inte till 1.0 for dessa Sampers-zoner:")
        print(bad)
    else:
        print(f"OK: {col} summerar till ~1.0 for alla Sampers-zoner med delning")

print("\nZonmappning:")
zone_key.head(20)

## 2. Las befintliga Dynameq-matriser (DQT-format)

In [ ]:
def parse_dqt(filepath):
    """Parsar en Dynameq DQT-fil och returnerar metadata + dict med DataFrames per tidssteg."""
    with open(filepath, "r", encoding="latin-1") as f:
        lines = f.readlines()

    # Parsa header
    metadata = {}
    metadata["raw_header_lines"] = []
    i = 0
    while i < len(lines):
        line = lines[i].rstrip("\n")
        if line == "DATA":
            metadata["raw_header_lines"].append(line)
            i += 1
            metadata["start_time"] = lines[i].rstrip("\n")
            metadata["raw_header_lines"].append(metadata["start_time"])
            i += 1
            metadata["end_time"] = lines[i].rstrip("\n")
            metadata["raw_header_lines"].append(metadata["end_time"])
            i += 1
            break
        else:
            metadata["raw_header_lines"].append(line)
            if line.startswith("* name:"):
                metadata["name"] = line.split("* name:")[1].strip()
            elif line.startswith("* units:"):
                metadata["units"] = line.split("* units:")[1].strip()
            elif line.startswith("VEH_CLASS"):
                metadata["raw_header_lines"].append(lines[i+1].rstrip("\n"))
                metadata["veh_class"] = lines[i+1].rstrip("\n")
                i += 2
                continue
            elif line.startswith("FORMAT:"):
                metadata["format"] = line.split("FORMAT:")[1].strip()
            i += 1

    # Parsa SLICE-block
    slices = {}  # time_label -> DataFrame
    slice_order = []
    while i < len(lines):
        line = lines[i].rstrip("\n")
        if line == "SLICE":
            i += 1
            time_label = lines[i].rstrip("\n")
            slice_order.append(time_label)
            i += 1
            # Zona-header (tab-separerad, borjar med tom cell)
            zone_header = lines[i].rstrip("\n").split("\t")
            zones = zone_header[1:]  # Forsta cellen ar tom
            i += 1
            # Las datarader
            data_rows = []
            row_zones = []
            while i < len(lines) and lines[i].rstrip("\n") != "SLICE":
                parts = lines[i].rstrip("\n").split("\t")
                row_zones.append(parts[0])
                data_rows.append([float(x) for x in parts[1:]])
                i += 1
            df = pd.DataFrame(data_rows, index=row_zones, columns=zones)
            slices[time_label] = df
        else:
            i += 1

    metadata["slice_order"] = slice_order
    metadata["zones"] = list(slices[slice_order[0]].columns)
    return metadata, slices

# Las PB och LB
pb_meta, pb_slices = parse_dqt(DYNAMEQ_DIR / "Nulage 2023 14-18 PB_matx.dqt")
lb_meta, lb_slices = parse_dqt(DYNAMEQ_DIR / "Nulage 2023 14-18 LB_matx.dqt")

print(f"PB: {pb_meta['name']}, {len(pb_slices)} tidssteg, {len(pb_meta['zones'])} zoner")
print(f"LB: {lb_meta['name']}, {len(lb_slices)} tidssteg, {len(lb_meta['zones'])} zoner")
print(f"Tidssteg: {pb_meta['slice_order']}")
print(f"Dynameq-zoner: {pb_meta['zones'][:10]}... ({len(pb_meta['zones'])} totalt)")

## 3. Las och summera Sampers OD-matriser

In [ ]:
def read_sampers_csv(filepath):
    """Las en Sampers OD-matris fran CSV. Returnerar DataFrame med zon-index."""
    df = pd.read_csv(filepath, index_col=0)
    df.index = df.index.astype(str)
    df.columns = df.columns.astype(str)
    return df

def sum_sampers_files(directory, exclude_files=None):
    """Summera alla mf*.csv filer i en mapp."""
    exclude_files = exclude_files or []
    csv_files = sorted([f for f in os.listdir(directory)
                       if f.startswith("mf") and f.endswith(".csv") and f not in exclude_files])
    print(f"  Filer att summera: {csv_files}")

    total = None
    for fname in csv_files:
        print(f"    Laser {fname}...", end=" ")
        df = read_sampers_csv(directory / fname)
        print(f"({df.shape[0]}x{df.shape[1]})")
        if total is None:
            total = df
        else:
            total = total.add(df, fill_value=0)
    return total

# Las och summera LB-matriser
print("Laser Sampers LB-matriser:")
sampers_lb = sum_sampers_files(SAMPERS_LB_DIR)
print(f"  Totalt LB: {sampers_lb.shape}, summa = {sampers_lb.values.sum():.1f} fordon\n")

# Las och summera PB-matriser (exkludera mo2.csv som inte ar relevant)
print("Laser Sampers PB-matriser:")
sampers_pb = sum_sampers_files(SAMPERS_PB_DIR)
print(f"  Totalt PB: {sampers_pb.shape}, summa = {sampers_pb.values.sum():.1f} fordon")

## 4. Zonkonvertering: Sampers -> Dynameq

- **Ursprungszoner (rader)**: fordelas proportionellt mot dagbefolkning
- **Destinationszoner (kolumner)**: fordelas proportionellt mot nattbefolkning
- Specialzoner 13, 14, 15 (ej i Sampers): skalas fran befintlig Dynameq-matris

In [ ]:
def convert_zones(sampers_matrix, zone_key, dynameq_zones):
    """
    Konvertera Sampers OD-matris till Dynameq zonsystem.

    For varje Sampers OD-par (orig_s, dest_s) med varde V:
      - Hitta alla Dynameq-zoner som orig_s mappar till, med dagbefolkningsandelar
      - Hitta alla Dynameq-zoner som dest_s mappar till, med nattbefolkningsandelar
      - Fordela: V * dag_andel_orig * natt_andel_dest
    """
    n_zones = len(dynameq_zones)
    result = pd.DataFrame(0.0, index=dynameq_zones, columns=dynameq_zones)

    # Bygg uppslag: sampers_zon -> lista av (dynameq_zon, dagbefolkning, nattbefolkning)
    sampers_to_dynameq = {}
    for _, row in zone_key.iterrows():
        s_zon = row["sampers_zon"]
        if s_zon not in sampers_to_dynameq:
            sampers_to_dynameq[s_zon] = []
        sampers_to_dynameq[s_zon].append({
            "dyn_zon": row["dynameq_zon"],
            "dag": row["dagbefolkning"],
            "natt": row["nattbefolkning"]
        })

    # Hitta unika Sampers-zoner som ar relevanta
    relevant_sampers_zones = set(sampers_to_dynameq.keys())
    available_orig = relevant_sampers_zones & set(sampers_matrix.index)
    available_dest = relevant_sampers_zones & set(sampers_matrix.columns)

    print(f"  Relevanta Sampers-zoner: {len(relevant_sampers_zones)}")
    print(f"  Tillgangliga som ursprung: {len(available_orig)}")
    print(f"  Tillgangliga som destination: {len(available_dest)}")

    missing_orig = relevant_sampers_zones - set(sampers_matrix.index)
    missing_dest = relevant_sampers_zones - set(sampers_matrix.columns)
    if missing_orig:
        print(f"  VARNING: Sampers-zoner saknas som ursprung: {missing_orig}")
    if missing_dest:
        print(f"  VARNING: Sampers-zoner saknas som destination: {missing_dest}")

    # Konvertera
    total_sampers_demand = 0
    total_dynameq_demand = 0
    for s_orig in available_orig:
        for s_dest in available_dest:
            val = sampers_matrix.loc[s_orig, s_dest]
            if val == 0:
                continue
            total_sampers_demand += val

            # Fordela pa Dynameq-zoner
            for d_orig in sampers_to_dynameq[s_orig]:
                for d_dest in sampers_to_dynameq[s_dest]:
                    contrib = val * d_orig["dag"] * d_dest["natt"]
                    if contrib > 0:
                        result.loc[d_orig["dyn_zon"], d_dest["dyn_zon"]] += contrib
                        total_dynameq_demand += contrib

    print(f"  Total Sampers-efterfragan (relevanta zoner): {total_sampers_demand:.1f}")
    print(f"  Total Dynameq-efterfragan efter konvertering: {total_dynameq_demand:.1f}")

    return result

print("Konverterar PB-matris...")
dynameq_zones = pb_meta["zones"]
pb_converted = convert_zones(sampers_pb, zone_key, dynameq_zones)
print(f"  Resultat: {pb_converted.shape}\n")

print("Konverterar LB-matris...")
lb_converted = convert_zones(sampers_lb, zone_key, dynameq_zones)
print(f"  Resultat: {lb_converted.shape}")

## 5. Tidskonvertering: Sampers 3h -> Dynameq kvartstimmar

Sampers-matriserna avser totaltrafik 15:00-18:00.
Dynameq behover kvartstimmatriser for 14:00-18:00.

1. Berakna tidsprofil fran befintlig Dynameq-matris
2. Applicera profilen pa nya volymer for 15:00-18:00
3. For 14:00-15:00: skala proportionellt mot forhallandet i befintlig data

In [ ]:
def compute_time_profile(slices, slice_order):
    """
    Berakna tidsprofil fran befintlig Dynameq-matris.
    Returnerar dict med tidssteg -> total volym (summa av alla OD-par).
    """
    profile = {}
    for t in slice_order:
        profile[t] = slices[t].values.sum()
    return profile

def apply_time_profile(converted_matrix, profile, slice_order, special_zone_slices=None):
    """
    Applicera tidsprofil pa konverterad matris.

    converted_matrix: total Sampers-efterfragan (fordon, 15:00-18:00) i Dynameq-zoner
    profile: dict tidssteg -> total volym fran befintlig Dynameq
    special_zone_slices: dict tidssteg -> DataFrame med specialzoners varden (for 13,14,15)

    Logik:
    - Sampers ger totalt antal fordon for 15:00-18:00
    - Dynameq behover veh/h per 15-minutersintervall
    - Skalningsfaktor = (Sampers_total / Dynameq_total_15_18_fordon) applicerad pa ALLA 16 tidssteg
      dar Dynameq_total_fordon = summa(veh/h * 0.25h)
    - Detta bevarar den relativa fordelningen mellan 14:00-15:00 och 15:00-18:00
    """
    # Identifiera 15:00-18:00 tidssteg (index 4-15 = "15:15" till "18:00")
    slices_15_18 = [t for t in slice_order if t >= "15:15"]  # 15:15, 15:30, ..., 18:00
    slices_14_15 = [t for t in slice_order if t < "15:15"]   # 14:15, 14:30, 14:45, 15:00

    print(f"  Tidssteg 14:00-15:00: {slices_14_15}")
    print(f"  Tidssteg 15:00-18:00: {slices_15_18}")

    # Total befintlig Dynameq-volym (veh/h) for 15:00-18:00
    total_dynameq_15_18_vehh = sum(profile[t] for t in slices_15_18)
    total_dynameq_14_15_vehh = sum(profile[t] for t in slices_14_15)

    # Konvertera till fordon: veh/h * 0.25h
    total_dynameq_15_18_veh = total_dynameq_15_18_vehh * 0.25
    total_dynameq_14_15_veh = total_dynameq_14_15_vehh * 0.25

    print(f"  Befintlig Dynameq 15-18 total: {total_dynameq_15_18_veh:.1f} fordon")
    print(f"  Befintlig Dynameq 14-15 total: {total_dynameq_14_15_veh:.1f} fordon")

    # Sampers total (fordon 15:00-18:00)
    sampers_total = converted_matrix.values.sum()
    print(f"  Ny Sampers 15-18 total: {sampers_total:.1f} fordon")

    if total_dynameq_15_18_veh > 0:
        # Skalningsfaktor
        scaling_factor = sampers_total / total_dynameq_15_18_veh
        print(f"  Skalningsfaktor (ny/gammal): {scaling_factor:.4f}")
    else:
        scaling_factor = 0
        print("  VARNING: Ingen befintlig trafik i 15-18 perioden!")

    # Berakna profilfraktioner for varje tidssteg
    # For 15:00-18:00: frakt_i = profile[i] / total_15_18_vehh
    # For 14:00-15:00: anvand samma skalningsfaktor (bevarar relativ fordelning)

    new_slices = {}
    for t in slice_order:
        if t in slices_15_18:
            # Profilfraktion for detta tidssteg
            if total_dynameq_15_18_vehh > 0:
                frac = profile[t] / total_dynameq_15_18_vehh
            else:
                frac = 1.0 / len(slices_15_18)
            # Nya fordon i detta 15-min intervall = sampers_total * frac
            # Konvertera till veh/h: fordon / 0.25h
            new_slice_vehh = converted_matrix * frac / 0.25
        else:
            # 14:00-15:00: skala befintlig profil
            # ratio = total_14_15 / total_15_18 i befintliga data
            if total_dynameq_15_18_vehh > 0 and total_dynameq_14_15_vehh > 0:
                frac_14 = profile[t] / total_dynameq_14_15_vehh
                ratio_14_to_15 = total_dynameq_14_15_veh / total_dynameq_15_18_veh
                # Nya fordon 14-15 = sampers_total * ratio
                new_14_15_total = sampers_total * ratio_14_to_15
                new_slice_vehh = converted_matrix * (ratio_14_to_15 * frac_14) / 0.25
            else:
                new_slice_vehh = converted_matrix * 0

        new_slices[t] = new_slice_vehh

    # Hantera specialzoner (13, 14, 15) - byt in befintliga varden skalade
    if special_zone_slices:
        special_dynameq_zones = list(special_zone_slices.keys())
        for t in slice_order:
            for sz in special_dynameq_zones:
                if sz in new_slices[t].index:
                    # Skala specialzonens rader och kolumner med skalningsfaktorn
                    for z in new_slices[t].columns:
                        new_slices[t].loc[sz, z] = special_zone_slices[sz][t]["row"].get(z, 0) * scaling_factor
                        new_slices[t].loc[z, sz] = special_zone_slices[sz][t]["col"].get(z, 0) * scaling_factor
                    # Diagonalen (sz, sz) skalas bara en gang
                    new_slices[t].loc[sz, sz] = special_zone_slices[sz][t]["diag"] * scaling_factor

    return new_slices

# Berakna tidsprofiler
pb_profile = compute_time_profile(pb_slices, pb_meta["slice_order"])
lb_profile = compute_time_profile(lb_slices, lb_meta["slice_order"])

print("Tidsprofil PB (total veh/h per tidssteg):")
for t, v in pb_profile.items():
    print(f"  {t}: {v:.1f}")
print(f"\nTidsprofil LB (total veh/h per tidssteg):")
for t, v in lb_profile.items():
    print(f"  {t}: {v:.1f}")

In [ ]:
# Extrahera specialzoners (13, 14, 15) befintliga varden for skalning
def extract_special_zone_data(slices, slice_order, special_zone_ids):
    """Extrahera rad-/kolumndata for specialzoner fran befintliga Dynameq-matriser."""
    result = {}
    for sz in special_zone_ids:
        result[sz] = {}
        for t in slice_order:
            df = slices[t]
            if sz in df.index:
                result[sz][t] = {
                    "row": df.loc[sz].to_dict(),    # Hela raden (fran sz till alla)
                    "col": df[sz].to_dict(),         # Hela kolumnen (fran alla till sz)
                    "diag": df.loc[sz, sz] if sz in df.columns else 0
                }
            else:
                result[sz][t] = {"row": {}, "col": {}, "diag": 0}
    return result

special_zone_ids = ["13", "14", "15"]

# Extrahera specialzonsdata
pb_special = extract_special_zone_data(pb_slices, pb_meta["slice_order"], special_zone_ids)
lb_special = extract_special_zone_data(lb_slices, lb_meta["slice_order"], special_zone_ids)

# Applicera tidsprofil pa PB
print("=== PB Tidskonvertering ===")
pb_new_slices = apply_time_profile(pb_converted, pb_profile, pb_meta["slice_order"], pb_special)

print(f"\n=== LB Tidskonvertering ===")
lb_new_slices = apply_time_profile(lb_converted, lb_profile, lb_meta["slice_order"], lb_special)

# Verifiering
print("\n=== Verifiering ===")
for label, new_slices, order in [("PB", pb_new_slices, pb_meta["slice_order"]),
                                   ("LB", lb_new_slices, lb_meta["slice_order"])]:
    total_veh = sum(new_slices[t].values.sum() * 0.25 for t in order)
    print(f"{label}: Totalt {total_veh:.1f} fordon over 14:00-18:00")

## 6. Skriv ut nya DQT-filer

Samma format som befintliga Dynameq-filer.

In [ ]:
def write_dqt(filepath, metadata, new_slices, new_name):
    """Skriv en Dynameq DQT-fil med samma format som originalet."""
    lines = []

    # Skriv header - anvand originalheadern men uppdatera namn
    from datetime import datetime
    now = datetime.now()
    date_str = now.strftime("%a %d. %b %H:%M:%S %Y")

    lines.append("<DYNAMEQ>")
    lines.append("<VERSION_23.0>")
    lines.append("<MATRIX_FILE>")
    lines.append(f"* CREATED {date_str}")
    lines.append(f"* name: {new_name}")

    # Behold description from original, skip the name/created/description lines
    for hl in metadata["raw_header_lines"]:
        if hl.startswith("* description:"):
            lines.append(hl)
            break

    # Check if there's an extra description line (PB has one)
    found_desc = False
    for idx, hl in enumerate(metadata["raw_header_lines"]):
        if hl.startswith("* description:"):
            found_desc = True
            # Check if next line is not a standard header line
            if idx + 1 < len(metadata["raw_header_lines"]):
                next_line = metadata["raw_header_lines"][idx + 1]
                if not next_line.startswith("*") and not next_line.startswith("FORMAT") and not next_line.startswith("VEH") and not next_line.startswith("DATA"):
                    lines.append(next_line)
            break

    if not found_desc:
        lines.append(f"* description: Prognos 2040, konverterad fran Sampers")

    lines.append(f"* units: {metadata['units']}")
    lines.append(f"FORMAT:{metadata['format']}")
    lines.append("VEH_CLASS")
    lines.append(metadata["veh_class"])
    lines.append("DATA")
    lines.append(metadata["start_time"])
    lines.append(metadata["end_time"])

    # Skriv SLICE-block
    zones = metadata["zones"]
    for t in metadata["slice_order"]:
        lines.append("SLICE")
        lines.append(t)
        # Zon-header
        lines.append("\t" + "\t".join(zones))
        # Datarader
        df = new_slices[t]
        for z in zones:
            values = []
            for z2 in zones:
                val = df.loc[z, z2]
                # Formatera: heltal om varden ar nara heltal, annars decimaler
                # Behall Dynameq-formatet: 0 for noll, annars upp till 6 decimaler
                if val == 0 or abs(val) < 1e-10:
                    values.append("0")
                else:
                    # Formatera med tillracklig precision, ta bort efterfoljande nollor
                    formatted = f"{val:.6g}"
                    values.append(formatted)
            lines.append(z + "\t" + "\t".join(values))

    # Skriv fil med latin-1 kodning (samma som original)
    with open(filepath, "w", encoding="latin-1", newline="\n") as f:
        f.write("\n".join(lines))
        f.write("\n")  # Avslutande radbrytning

    print(f"Fil skriven: {filepath}")
    print(f"  Storlek: {os.path.getsize(filepath) / 1024:.1f} KB")

# Skriv PB
pb_output_path = OUTPUT_DIR / "Prognos 2040 14-18 PB_matx.dqt"
write_dqt(pb_output_path, pb_meta, pb_new_slices, "Prognos 2040 14-18 PB")

# Skriv LB
lb_output_path = OUTPUT_DIR / "Prognos 2040 14-18 LB_matx.dqt"
write_dqt(lb_output_path, lb_meta, lb_new_slices, "Prognos 2040 14-18 LB")

print("\nKlar! Filer sparade i:", OUTPUT_DIR)

## 7. Visualisering och rimlighetsanalys

Jamfor befintliga (2023) och nya (2040) matriser.

In [ ]:
### 7a. Tidsprofil-jamforelse

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, label, old_profile, new_slices, order in [
    (axes[0], "PB", pb_profile, pb_new_slices, pb_meta["slice_order"]),
    (axes[1], "LB", lb_profile, lb_new_slices, lb_meta["slice_order"])
]:
    old_vals = [old_profile[t] for t in order]
    new_vals = [new_slices[t].values.sum() for t in order]
    x = range(len(order))

    ax.bar([i - 0.2 for i in x], old_vals, 0.4, label="2023 (befintlig)", alpha=0.8, color="steelblue")
    ax.bar([i + 0.2 for i in x], new_vals, 0.4, label="2040 (ny)", alpha=0.8, color="coral")
    ax.set_xticks(list(x))
    ax.set_xticklabels(order, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Total veh/h")
    ax.set_title(f"{label} - Tidsprofil 14:00-18:00")
    ax.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "tidsprofil_jamforelse.png", dpi=150, bbox_inches="tight")
plt.show()
print("Sparad: tidsprofil_jamforelse.png")

In [ ]:
### 7b. Totaler per zon - jamforelse av produktion (utflode) och attraktion (inflode)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for col, (label, old_slices, new_slices, order) in enumerate([
    ("PB", pb_slices, pb_new_slices, pb_meta["slice_order"]),
    ("LB", lb_slices, lb_new_slices, lb_meta["slice_order"])
]):
    zones = pb_meta["zones"]

    # Summera over alla tidssteg (konvertera till fordon)
    old_prod = np.zeros(len(zones))  # Produktion = summa radvis
    old_attr = np.zeros(len(zones))  # Attraktion = summa kolumnvis
    new_prod = np.zeros(len(zones))
    new_attr = np.zeros(len(zones))

    for t in order:
        old_prod += old_slices[t].values.sum(axis=1) * 0.25
        old_attr += old_slices[t].values.sum(axis=0) * 0.25
        new_prod += new_slices[t].values.sum(axis=1) * 0.25
        new_attr += new_slices[t].values.sum(axis=0) * 0.25

    # Produktion
    ax = axes[0, col]
    ax.scatter(old_prod, new_prod, alpha=0.6, s=20, color="steelblue")
    max_val = max(old_prod.max(), new_prod.max()) * 1.1
    ax.plot([0, max_val], [0, max_val], "r--", linewidth=1, label="1:1")
    ax.set_xlabel("2023 produktion (fordon)")
    ax.set_ylabel("2040 produktion (fordon)")
    ax.set_title(f"{label} - Produktion per zon")
    ax.legend()

    # Attraktion
    ax = axes[1, col]
    ax.scatter(old_attr, new_attr, alpha=0.6, s=20, color="coral")
    max_val = max(old_attr.max(), new_attr.max()) * 1.1
    ax.plot([0, max_val], [0, max_val], "r--", linewidth=1, label="1:1")
    ax.set_xlabel("2023 attraktion (fordon)")
    ax.set_ylabel("2040 attraktion (fordon)")
    ax.set_title(f"{label} - Attraktion per zon")
    ax.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "produktion_attraktion_jamforelse.png", dpi=150, bbox_inches="tight")
plt.show()
print("Sparad: produktion_attraktion_jamforelse.png")

In [ ]:
### 7c. OD-matris heatmap - summerad over hela perioden

fig, axes = plt.subplots(2, 2, figsize=(16, 14))

for col, (label, old_slices, new_slices, order) in enumerate([
    ("PB", pb_slices, pb_new_slices, pb_meta["slice_order"]),
    ("LB", lb_slices, lb_new_slices, lb_meta["slice_order"])
]):
    # Summera alla tidssteg
    old_total = sum(old_slices[t] for t in order) * 0.25
    new_total = sum(new_slices[t] for t in order) * 0.25

    vmax = max(old_total.values.max(), new_total.values.max())

    ax = axes[0, col]
    sns.heatmap(old_total.values, ax=ax, cmap="YlOrRd", vmin=0, vmax=vmax,
                xticklabels=False, yticklabels=False)
    ax.set_title(f"{label} 2023 (befintlig)")
    ax.set_xlabel("Destination")
    ax.set_ylabel("Ursprung")

    ax = axes[1, col]
    sns.heatmap(new_total.values, ax=ax, cmap="YlOrRd", vmin=0, vmax=vmax,
                xticklabels=False, yticklabels=False)
    ax.set_title(f"{label} 2040 (ny)")
    ax.set_xlabel("Destination")
    ax.set_ylabel("Ursprung")

plt.suptitle("OD-matriser: Total trafik 14:00-18:00 (fordon)", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "od_heatmap_jamforelse.png", dpi=150, bbox_inches="tight")
plt.show()
print("Sparad: od_heatmap_jamforelse.png")

In [ ]:
### 7d. Sammanfattningstabell

zones = pb_meta["zones"]
order = pb_meta["slice_order"]

summary_data = []
for label, old_slices, new_slices in [
    ("PB", pb_slices, pb_new_slices),
    ("LB", lb_slices, lb_new_slices)
]:
    old_total_veh = sum(old_slices[t].values.sum() * 0.25 for t in order)
    new_total_veh = sum(new_slices[t].values.sum() * 0.25 for t in order)
    change_pct = ((new_total_veh / old_total_veh) - 1) * 100 if old_total_veh > 0 else float('inf')

    old_15_18 = sum(old_slices[t].values.sum() * 0.25 for t in order if t >= "15:15")
    new_15_18 = sum(new_slices[t].values.sum() * 0.25 for t in order if t >= "15:15")
    old_14_15 = sum(old_slices[t].values.sum() * 0.25 for t in order if t < "15:15")
    new_14_15 = sum(new_slices[t].values.sum() * 0.25 for t in order if t < "15:15")

    # Antal OD-par med trafik
    old_nonzero = sum((old_slices[t].values > 0).sum() for t in order)
    new_nonzero = sum((new_slices[t].values > 0).sum() for t in order)

    summary_data.append({
        "Typ": label,
        "2023 total (fdn)": f"{old_total_veh:.0f}",
        "2040 total (fdn)": f"{new_total_veh:.0f}",
        "Forandring (%)": f"{change_pct:+.1f}%",
        "2023 14-15 (fdn)": f"{old_14_15:.0f}",
        "2040 14-15 (fdn)": f"{new_14_15:.0f}",
        "2023 15-18 (fdn)": f"{old_15_18:.0f}",
        "2040 15-18 (fdn)": f"{new_15_18:.0f}",
        "2023 OD-par>0": f"{old_nonzero}",
        "2040 OD-par>0": f"{new_nonzero}",
    })

summary_df = pd.DataFrame(summary_data)
print("=== SAMMANFATTNING ===")
print(summary_df.to_string(index=False))
summary_df

In [ ]:
### 7e. Differens-heatmap (2040 - 2023)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for col, (label, old_slices, new_slices, order) in enumerate([
    ("PB", pb_slices, pb_new_slices, pb_meta["slice_order"]),
    ("LB", lb_slices, lb_new_slices, lb_meta["slice_order"])
]):
    old_total = sum(old_slices[t] for t in order) * 0.25
    new_total = sum(new_slices[t] for t in order) * 0.25
    diff = new_total - old_total

    vmax = max(abs(diff.values.min()), abs(diff.values.max()))

    ax = axes[col]
    sns.heatmap(diff.values, ax=ax, cmap="RdBu_r", center=0, vmin=-vmax, vmax=vmax,
                xticklabels=False, yticklabels=False)
    ax.set_title(f"{label}: Differens (2040 - 2023)")
    ax.set_xlabel("Destination")
    ax.set_ylabel("Ursprung")

plt.suptitle("Forandring i OD-matris (fordon, 14:00-18:00)", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "od_differens_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Sparad: od_differens_heatmap.png")

## 8. Ladda ner filer

Kör cellen nedan för att visa sökvägar till output-filerna.

In [ ]:
# Visa output-filer
print("=== Genererade filer ===")
for f in sorted(OUTPUT_DIR.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:45s} {size_kb:8.1f} KB")

print(f"\nFullstandig sokvag: {OUTPUT_DIR.resolve()}")

# For Jupyter - skapa nedladdningslankar om IPython finns
try:
    from IPython.display import display, FileLink, HTML
    print("\nKlicka for att ladda ner:")
    for f in sorted(OUTPUT_DIR.iterdir()):
        if f.suffix == ".dqt":
            display(FileLink(str(f), result_html_prefix="  "))
except ImportError:
    print("\n(Kor i Jupyter for interaktiva nedladdningslankar)")